In [ ]:
from google.colab import files
uploaded = files.upload()

Saving nba_2022-23_all_stats_with_salary.csv to nba_2022-23_all_stats_with_salary.csv


In [ ]:
import os
print(os.listdir())

['.config', 'nba_2022-23_all_stats_with_salary.csv', 'sample_data']


In [ ]:
import pandas as pd

df = pd.read_csv('nba_2022-23_all_stats_with_salary.csv')
print(df.shape)
print(df.columns.tolist())

(467, 52)
['Unnamed: 0', 'Player Name', 'Salary', 'Position', 'Age', 'Team', 'GP', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Total Minutes', 'PER', 'TS%', '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP']


In [ ]:
print(df.columns.tolist())

['Unnamed: 0', 'Player Name', 'Salary', 'Position', 'Age', 'Team', 'GP', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Total Minutes', 'PER', 'TS%', '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP']


In [ ]:
print(df[['Player Name', 'Salary', 'PTS', 'AST', 'TRB', 'PER']].head(10))
print(f"\nNull values in Salary: {df['Salary'].isna().sum()}")
print(f"\nSalary range: ${df['Salary'].min():,.0f} to ${df['Salary'].max():,.0f}")

             Player Name    Salary   PTS  AST   TRB   PER
0          Stephen Curry  48070014  29.4  6.3   6.1  24.1
1              John Wall  47345760  11.4  5.2   2.7  13.6
2      Russell Westbrook  47080179  15.9  7.5   5.8  16.1
3           LeBron James  44474988  28.9  6.8   8.3  23.9
4           Kevin Durant  44119845  29.1  5.0   6.7  25.9
5           Bradley Beal  43279250  23.2  5.4   3.9  19.7
6          Kawhi Leonard  42492492  23.8  3.9   6.5  23.9
7            Paul George  42492492  23.8  5.1   6.1  19.6
8  Giannis Antetokounmpo  42492492  31.1  5.7  11.8  29.0
9         Damian Lillard  42492492  32.2  7.3   4.8  26.7

Null values in Salary: 0

Salary range: $5,849 to $48,070,014


In [ ]:
df_clean = df.dropna(subset=['Salary', 'PTS', 'AST', 'TRB'])

df_clean = df_clean[df_clean['GP'] >= 20]

df_clean['Salary_M'] = df_clean['Salary'] / 1000000  # Salary in millions
df_clean['Points_per_Million'] = df_clean['PTS'] / df_clean['Salary_M']
df_clean['Value_Score'] = (df_clean['PTS'] + df_clean['AST'] + df_clean['TRB']) / df_clean['Salary_M']

print(f"Players after filtering: {len(df_clean)}")
print(df_clean[['Player Name', 'Salary_M', 'PTS', 'Value_Score']].sort_values('Value_Score', ascending=False).head(10))

Players after filtering: 376
          Player Name  Salary_M   PTS  Value_Score
391         Kris Dunn  1.000001  13.2    23.299977
430  Orlando Robinson  0.386055   3.7    22.276619
418        Luka Garza  0.508891   6.5    18.471539
400      Anthony Lamb  0.694878   6.7    16.837488
422   Dominick Barlow  0.508891   3.9    16.506482
309      Desmond Bane  2.130240  21.5    14.505408
419      Kevon Harris  0.508891   4.1    13.165884
353         Tre Jones  1.782621  12.9    12.958447
370     Austin Reaves  1.563518  13.0    12.407916
385       Jaden Hardy  1.017781   8.8    11.888609


In [ ]:
position_salary = df_clean.groupby('Position').agg(
    avg_salary=('Salary_M', 'mean'),
    avg_points=('PTS', 'mean'),
    avg_value=('Value_Score', 'mean'),
    player_count=('Player Name', 'count')
).reset_index()

print(position_salary)

  Position  avg_salary  avg_points  avg_value  player_count
0        C    8.085055    8.042308   3.835663            78
1       PF   10.358642   10.678571   3.139598            70
2       PG   13.712422   12.042623   3.366520            61
3    PG-SG   21.458528   16.050000   1.634363             2
4       SF   10.347190   10.624638   3.297582            69
5    SF-PF    3.000000    6.600000   3.300000             1
6    SF-SG   12.239763   12.100000   1.467882             2
7       SG    8.203737   10.682418   3.632744            91
8    SG-PG   16.650807   11.750000   1.161150             2


In [ ]:
team_summary = df_clean.groupby('Team').agg(
    total_payroll=('Salary_M', 'sum'),
    avg_salary=('Salary_M', 'mean'),
    avg_points=('PTS', 'mean'),
    avg_value=('Value_Score', 'mean'),
    player_count=('Player Name', 'count')
).reset_index()

top_paid = df_clean.nlargest(20, 'Salary_M')[['Player Name', 'Team', 'Position', 'Salary_M', 'PTS', 'AST', 'TRB', 'Value_Score']]

top_value = df_clean.nlargest(20, 'Value_Score')[['Player Name', 'Team', 'Position', 'Salary_M', 'PTS', 'AST', 'TRB', 'Value_Score']]

df_export = df_clean[['Player Name', 'Team', 'Position', 'GP', 'PTS', 'AST', 'TRB',
                       'Salary_M', 'Points_per_Million', 'Value_Score', 'PER']]

print("All summaries created!")

All summaries created!


In [ ]:
df_export.to_csv('nba_players_clean.csv', index=False)
position_salary.to_csv('position_salary.csv', index=False)
team_summary.to_csv('team_summary.csv', index=False)
top_paid.to_csv('top_paid.csv', index=False)
top_value.to_csv('top_value.csv', index=False)

from google.colab import files
for f in ['nba_players_clean.csv', 'position_salary.csv',
          'team_summary.csv', 'top_paid.csv', 'top_value.csv']:
    files.download(f)

print("All files exported!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files exported!


In [ ]:
df_export.to_csv('/content/nba_players_clean.csv', index=False)
print("File saved!")

File saved!
